# EDA - Memorias RAM PcComponentes

## Instrucciones para ejecutar
1. Ejecuta las celdas **en orden** (de arriba hacia abajo).
2. La celda de carga configura el path automáticamente usando `config/settings.py`.
3. Verás impresiones de las rutas y los resultados (prints + gráficos) al ejecutar cada celda de código.
4. Las explicaciones están en las celdas Markdown que siguen a cada análisis.

**Dependencias** (si hace falta):
```python
# !pip install pandas matplotlib seaborn numpy
```

## Objetivo
Explorar los datos brutos extraídos de PcComponentes para memorias RAM (~1646 productos).

Este análisis nos ayudará a:
- Entender la distribución del catálogo actual
- Identificar campos con problemas de extracción (ej. voltaje)
- Descubrir patrones en precio, especificaciones y opiniones
- Definir features útiles de cara al clustering futuro (Fase 02)
- Detectar oportunidades de mejora en los scrapers

**Datos utilizados:** `data/brutos/ram/listado_ram.json` + `detalle_ram.json` (rutas desde config/settings.py)


## 1. Carga de datos y preparación


In [ ]:
# ============================================================
# CELDA 1: CARGA DE DATOS Y PREPARACIÓN DEL ENTORNO
# ============================================================
# - Configura el path del proyecto de forma robusta (ejecutable desde notebooks/ o raíz)
# - Carga las rutas centralizadas desde config/settings.py
# - Lee los JSON brutos generados por los scrapers (listado + detalle)
# - Crea DataFrames base para el análisis posterior
# ============================================================

import json
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo global para los gráficos del notebook
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
%matplotlib inline


# ------------------------------------------------------------
# Detección robusta de la raíz del proyecto
# Busca hacia arriba en la jerarquía de carpetas hasta encontrar config/settings.py
# ------------------------------------------------------------
def get_project_root():
    """Devuelve la ruta raíz del proyecto buscando config/settings.py hacia arriba."""
    current = Path.cwd().resolve()
    for parent in [current] + list(current.parents):
        if (parent / "config" / "settings.py").exists():
            return parent
    # Fallback razonable si se ejecuta desde notebooks/
    return current.parent if current.name == "notebooks" else current


# Añadir raíz al sys.path para poder importar config.settings
project_root = get_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Rutas oficiales de los archivos de datos RAM (centralizadas en settings)
from config.settings import RAM_LISTADO_JSON, RAM_DETALLE_JSON

listado_path = RAM_LISTADO_JSON
detalle_path = RAM_DETALLE_JSON

print("Cargando desde:")
print("  -", listado_path)
print("  -", detalle_path)

# Carga de JSONs generados por scraper (listado resumido + detalle enriquecido)
with open(listado_path, encoding="utf-8") as f:
    listado = json.load(f)

with open(detalle_path, encoding="utf-8") as f:
    detalle = json.load(f)

# DataFrames principales que usaremos en todo el EDA
productos_listado = pd.DataFrame(listado["productos"])
productos_detalle = pd.DataFrame(detalle["productos"])

print("\nListado productos:", len(productos_listado))
print("Detalle productos:", len(productos_detalle))
print("Errores:", len(detalle.get("errores", [])))


## 2. Resumen General del Dataset

- **Productos en listado**: 1646
- **Productos con detalle completo**: 1574
- **Total reseñas recolectadas**: ~5.876 (aprox 3.7 por producto)

Nota: Hay una ligera diferencia entre listado y detalle. Algunos productos no devolvieron ficha completa en el momento del scraping.


## 3. Análisis de Precios


In [ ]:
# ============================================================
# CELDA 2: ANÁLISIS DE PRECIOS
# ============================================================
# - Unimos el detalle (precio real de ficha) con datos del listado (precio, valoración, nº opiniones)
# - Estadísticas descriptivas + histograma de distribución de precios
# ============================================================

# Merge left para conservar todos los productos con detalle y enriquecer con campos del listado
df = productos_detalle.merge(
    productos_listado[["id", "precio_listado", "valoracion_listado", "num_opiniones_listado"]], 
    on="id", how="left"
)

# Estadísticas básicas de precio (media, std, min, quartiles, max)
print("=== Estadísticas de precio ===")
print(df["precio"].describe())

# Detección rápida de outliers de precio
print("\nProductos con precio < 50€:", (df["precio"] < 50).sum())
print("Productos con precio > 2000€:", (df["precio"] > 2000).sum())

# Histograma simple para visualizar la distribución
df["precio"].hist(bins=50)
plt.title("Distribución de precios de kits de RAM")
plt.xlabel("Precio (€)")
plt.ylabel("Número de productos")
plt.show()


**Observaciones precios:**
- Rango muy amplio (desde ~1€ hasta kits de gama alta >7000€).
- Mediana alrededor de 390€ → muchos kits de 32GB DDR5 de gama media-alta.
- Outliers caros suelen ser configuraciones de 128GB+ o muy alta frecuencia.


## 4. Especificaciones Técnicas


In [ ]:
# ============================================================
# CELDA 3: ESPECIFICACIONES TÉCNICAS (parte 1)
# ============================================================
# - Normalizamos el campo JSON "especificaciones" a columnas planas
# - Analizamos tipo de memoria (DDR4 vs DDR5), capacidad y primeros problemas de parseo
# ============================================================

# Desanidar el diccionario de especificaciones técnicas extraído por el scraper
especs = pd.json_normalize(df["especificaciones"])

# --- Tipo de memoria ---
print("=== Tipo de memoria ===")
print(especs["tipo_memoria"].value_counts(dropna=False))

# --- Capacidad total del kit (en GB) ---
# Usamos to_numeric con errors='coerce' para convertir fallos a NaN
capacidad = pd.to_numeric(especs["capacidad_gb"], errors="coerce")
print("\n=== Capacidad total (GB) ===")
print(capacidad.describe())
print("Valores sospechosamente altos (>1000):", (capacidad > 1000).sum())


In [ ]:
# ============================================================
# CELDA 4: ESPECIFICACIONES TÉCNICAS (parte 2 - frecuencia y latencia)
# ============================================================
# - Frecuencia en MHz (importante para rendimiento)
# - Latencia CL (Column Latency) - valores más comunes y nulos
# ============================================================

# --- Frecuencia ---
frec = pd.to_numeric(especs["frecuencia_mhz"], errors="coerce")
print("=== Frecuencia (MHz) ===")
print(frec.describe())

# --- Latencia CL ---
print("\n=== Latencia CL más comunes ===")
print(especs["latencia_cl"].value_counts(dropna=False).head(10))


**Hallazgos importantes en specs:**
- DDR5 ya es mayoritario (824 vs 580 DDR4).
- Muchos valores de capacidad parecen mal parseados (números enormes — problema de extracción).
- Frecuencia media ~4570 MHz, con pico fuerte en 5200-6000 MHz (típico DDR5 gaming).
- Latencia CL tiene bastantes nulos. Los valores más comunes son 40, 16, 22, 36.


## 5. El problema del Voltaje (campo crítico)


In [ ]:
# ============================================================
# CELDA 5: PROBLEMA CRÍTICO - CAMPO VOLTAJE
# ============================================================
# El voltaje es una especificación clave para compatibilidad y estabilidad.
# Aquí detectamos que no se está extrayendo (problema conocido en To_Do_List).
# ============================================================

print("Voltaje nulos:", especs["voltaje"].isna().sum(), "/", len(especs))
print("Valores únicos de voltaje:", especs["voltaje"].unique()[:5])


**Conclusión clara:** El campo `voltaje` no se está extrayendo en absoluto (100% nulo).  

Esto coincide con el To_Do_List. Es una prioridad de mejora en el parser de `detalle_ram.py` o mediante re-procesado de las fichas HTML guardadas si las tenemos.


## 6. Reseñas y Valoraciones de usuarios


In [ ]:
# ============================================================
# CELDA 6: RESEÑAS Y VALORACIONES DE USUARIOS
# ============================================================
# - Aplanamos la lista de reseñas embebidas en cada producto del detalle
# - Añadimos producto_id para poder relacionarlas después
# - Métricas de cobertura de pros/contras y distribución de estrellas
# NOTA: Actualmente son solo una muestra (JSON-LD embebido), no todas las reseñas.
# ============================================================

all_resenas = []
for p in detalle["productos"]:
    resenas_producto = p.get("resenas") or []
    for r in resenas_producto:
        # Creamos una copia para no mutar el JSON original cargado en memoria
        r_copia = dict(r)
        r_copia["producto_id"] = p["id"]
        all_resenas.append(r_copia)

resenas_df = pd.DataFrame(all_resenas)

print(f"Total reseñas recolectadas: {len(resenas_df)}")
print(f"Reseñas por producto (media): {len(resenas_df) / len(detalle['productos']):.1f}")

print("\nDistribución de estrellas en reseñas individuales:")
print(resenas_df["valoracion"].value_counts(dropna=False).sort_index())

print("\n% con pros:", round(resenas_df["pros"].notna().mean()*100, 1))
print("% con contras:", round(resenas_df["contras"].notna().mean()*100, 1))


**Observaciones reseñas:**
- Fuerte sesgo positivo: la inmensa mayoría son 5 estrellas (4352 de ~5876).
- Solo ~3.7 reseñas por producto de media (recordemos que son solo una muestra embebida en JSON-LD, no todas las opiniones del producto).
- Pros y contras están presentes en la mayoría de las reseñas recolectadas → material excelente para NLP y extracción de temas.
- El campo `opinion_verificada` parece no haberse extraído (todo nulo en los datos actuales).


## 7. Marcas más comunes


In [ ]:
# ============================================================
# CELDA 7: EXTRACCIÓN SIMPLE DE MARCA DESDE EL NOMBRE
# ============================================================
# Función heurística para inferir marca a partir del nombre del producto.
# Se usa para análisis rápido de participación de mercado.
# En el futuro se debería normalizar desde el scraper o con un catálogo maestro.
# ============================================================

def extraer_marca(nombre):
    """Extrae la marca conocida más probable del nombre del producto (upper case match)."""
    nombre = str(nombre).upper()
    # Orden intencional: primero las más específicas / frecuentes
    marcas_conocidas = [
        "KINGSTON", "CORSAIR", "G.SKILL", "CRUCIAL", "SAMSUNG", "TEAMGROUP",
        "PATRIOT", "ADATA", "LEXAR", "KLEVV", "FORGEON", "XPG", "GOODRAM", "PNY"
    ]
    for marca in marcas_conocidas:
        if marca in nombre:
            return marca
    return "OTROS / DESCONOCIDA"


# Aplicamos la función y mostramos el conteo de las principales
df["marca"] = df["nombre"].apply(extraer_marca)
print(df["marca"].value_counts().head(12))


Marcas dominantes: Kingston, G.Skill y Corsair lideran claramente. Hay un volumen importante de "OTROS".


## 8. Calidad de los datos y campos problemáticos


### Campos con problemas detectados:
- **voltaje**: 100% nulo → prioridad alta de mejora en scraping
- **latencia_cl**: bastantes nulos (~18%)
- **capacidad_gb**: algunos valores absurdamente altos (problema de parseo en el scraper)
- **valoracion_media / num_opiniones**: en muchos casos aparecen como 0 (posible bug en la extracción del bloque de opiniones)
- **opinion_verificada**: no se está capturando

### Diferencia listado vs detalle
Hay unos 72 productos que están en el listado pero no tienen detalle completo. Merece la pena investigar (posibles bloqueos puntuales o productos eliminados entre runs).


## 9. Conclusiones y Recomendaciones


### Lo bueno
- Buen volumen de datos (1646 productos + casi 6k reseñas).
- Cobertura decente de DDR5 (ya representa la mayoría del catálogo actual).
- Pros/contras disponibles en la mayoría de reseñas → material excelente para análisis de sentimiento y extracción de temas (Fase 02).
- El sistema de scraping parece robusto (0 errores reportados en el JSON).

### Áreas de mejora urgentes (antes de modelado / DB)
1. **Reparar extracción de voltaje** (crítico para especificaciones técnicas reales).
2. **Mejorar parsing de capacidad_gb y latencia_cl** (hay basura numérica en los datos).
3. **Revisar extracción de valoraciones/resumen de opiniones** (muchos ceros sospechosos).
4. **Evaluar re-run selectivo** de productos con datos incompletos o enriquecer con sección completa de opiniones (actualmente solo muestra embebida).
5. **Normalizar marcas y modelos** para futuro clustering.

### Features interesantes para clustering (Fase 02)
- Capacidad total + num_módulos + capacidad_por_módulo
- Frecuencia + Latencia (CL)
- Tipo de memoria (DDR4 / DDR5)
- Precio (quizá log o por GB)
- Score de sentimiento de reseñas (una vez tengamos NLP)
- Presencia de disipador / RGB (campo 'diseno')
- Marca (one-hot o embedding)

### Próximos pasos sugeridos (sin tocar DB todavía)
- Crear scripts de post-procesado que limpien/enriquezcan los JSONs actuales.
- Completar la documentación y scrapers de Tarjetas Gráficas.
- Empezar a prototipar funciones de normalización de especificaciones en `pipeline/`.
- Seguir con EDA más profunda (correlaciones precio-specs, análisis de texto de pros/contras).
